# Pumpkin Seed Classification with a Random Forest

This notebook develops and evaluates a **binary classification** model that predicts a pumpkin seed's variety from measured physical characteristics. The example emphasizes a reproducible machine-learning workflow: inspect the data, prepare the outcome, split the observations, train the model, evaluate out-of-sample predictions, and interpret feature importance.

**Learning objectives**

By the end of the notebook, you should be able to:

1. distinguish predictors from a categorical target;
2. encode class labels for use with scikit-learn;
3. create training and test sets without leaking information;
4. fit a Random Forest classifier;
5. interpret a confusion matrix and common classification metrics; and
6. explain what Random Forest feature importance does—and does not—tell us.

**Source paper:** [Classification of pumpkin seeds using machine learning methods](https://link.springer.com/article/10.1007/s10722-021-01226-0)


## 1. Setup

Import the libraries used throughout the analysis. Keeping imports in one place makes the notebook easier to audit and rerun.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

RANDOM_STATE = 96
sns.set_theme(style="whitegrid")


## 2. Import and inspect the data

The path below assumes the notebook is running in Google Colab with Google Drive mounted. If your Drive is mounted elsewhere, update `DATA_PATH` only; the rest of the notebook can remain unchanged.

Before modeling, inspect the dimensions, column types, missing values, and class distribution. These checks help catch problems such as an incorrect file, nonnumeric predictors, or an imbalanced target.


In [ ]:
path = ""

df = pd.read_excel(path)
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
df.head()


In [ ]:
df.info()

missing_values = df.isna().sum().rename("missing_count")
missing_values.to_frame()


### Examine the target distribution

`Class` is the outcome the model will predict. A count plot shows whether one class is much more common than the other. Severe imbalance can make accuracy misleading because a model may score well simply by favoring the majority class.


In [ ]:
class_counts = df["Class"].value_counts()
display(class_counts.to_frame(name="count"))

ax = sns.countplot(data=df, x="Class", hue="Class", legend=False)
ax.set(title="Pumpkin seed observations by class", xlabel="Class", ylabel="Count")
plt.show()


## 3. Encode the target and define the predictors

Scikit-learn estimators can work with string class labels, but numeric encoding makes the metric calculations below explicit. `LabelEncoder` assigns integers in alphabetical order. Saving the mapping is essential: otherwise, a result such as `1` has no substantive meaning.

The predictors are selected by **name** rather than by position. This is safer than assuming the target will always be the final column.


In [ ]:
label_encoder = LabelEncoder()
df["Class_encoded"] = label_encoder.fit_transform(df["Class"])

class_mapping = dict(
    zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))
)
print("Class mapping:", class_mapping)

X = df.drop(columns=["Class", "Class_encoded"])
y = df["Class_encoded"]

print(f"Predictor matrix: {X.shape}")
print(f"Target vector: {y.shape}")
X.head()


## 4. Split the data into training and test sets

The model learns patterns from the **training set**. The untouched **test set** provides an estimate of performance on new observations.

- `test_size=0.20` reserves 20% of the observations for evaluation.
- `random_state` makes the split reproducible.
- `stratify=y` preserves approximately the same class proportions in both subsets.

The test set should not influence preprocessing choices, model fitting, or hyperparameter selection.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame(
    {
        "observations": [len(X_train), len(X_test)],
        "positive_class_share": [y_train.mean(), y_test.mean()],
    },
    index=["Training", "Test"],
)
split_summary


## 5. Standardize the predictors

Standardization subtracts each feature's training-set mean and divides by its training-set standard deviation. The result is a feature with mean near 0 and standard deviation near 1.

**Important:** Random Forests are tree-based and generally do **not** require feature scaling because their splits depend on feature order, not distance. It is retained here to demonstrate a reusable preprocessing pattern. The scaler is fit only on `X_train`, then applied to `X_test`; fitting on all observations would leak information from the test set.


In [ ]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

X_train_scaled.agg(["mean", "std"]).round(2)


## 6. Train a Random Forest classifier

A Random Forest combines predictions from many decision trees. Each tree is trained on a bootstrap sample and considers a random subset of predictors at each split. Aggregating many varied trees typically reduces the instability and overfitting of a single decision tree.

Here, `n_estimators=100` creates 100 trees. The random seed ensures that the same forest is produced each time the notebook is run.


In [ ]:
pumpkin_classifier = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
)

pumpkin_classifier.fit(X_train_scaled, y_train)


## 7. Generate out-of-sample predictions

Predictions are made only for the held-out test observations. The comparison table translates the numeric codes back to the original class names and flags incorrect predictions for easy inspection.


In [ ]:
y_pred = pumpkin_classifier.predict(X_test_scaled)

results = pd.DataFrame(
    {
        "Actual": label_encoder.inverse_transform(y_test.to_numpy()),
        "Predicted": label_encoder.inverse_transform(y_pred),
    },
    index=y_test.index,
).sort_index()
results["Correct"] = results["Actual"] == results["Predicted"]

results.head(10)


> **Optional export:** Uncomment the next line if you want to save the test-set predictions. `index=False` prevents pandas from writing the DataFrame index as an extra column.


In [ ]:
# results.to_csv("pumpkin_test_predictions.csv", index=False)


## 8. Confusion matrix

A confusion matrix cross-tabulates actual and predicted classes. Correct predictions appear on the diagonal; off-diagonal values are classification errors. Reading the original class names on both axes is more informative than reading the encoded values `0` and `1`.


In [ ]:
class_names = label_encoder.classes_
cm = confusion_matrix(y_test, y_pred)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names,
)
display.plot(cmap="Blues", colorbar=False)
display.ax_.set_title("Random Forest confusion matrix")
plt.grid(False)
plt.show()


## 9. Evaluate classification performance

No single metric tells the whole story:

- **Accuracy** is the share of all test observations classified correctly.
- **Precision** asks: among observations predicted as the positive class, how many were correct?
- **Recall** asks: among actual positive-class observations, how many did the model identify?
- **F1 score** is the harmonic mean of precision and recall; it is useful when both error types matter.

For the binary metrics below, the positive class is the label encoded as `1`. The code prints that class name explicitly so the interpretation is unambiguous.


In [ ]:
positive_class = label_encoder.inverse_transform([1])[0]
print(f"Positive class for precision, recall, and F1: {positive_class}")

metrics = pd.Series(
    {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 score": f1_score(y_test, y_pred, zero_division=0),
    },
    name="score",
)
metrics.to_frame().round(3)


The classification report provides precision, recall, and F1 separately for each class. The **macro average** gives both classes equal weight, while the **weighted average** weights each class by its number of observations.


In [ ]:
report = classification_report(
    y_test,
    y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(report).T.round(3)


## 10. Inspect feature importance

Random Forest impurity-based importance measures how much splits using each feature reduce node impurity across the forest. The values are relative and sum to 1.

Interpret them cautiously:

- importance is **not** a causal effect;
- correlated predictors may divide importance between them; and
- continuous or high-cardinality predictors can receive more importance.

Permutation importance on held-out data is a useful follow-up when more rigorous interpretation is needed.


In [ ]:
feature_importance_df = (
    pd.DataFrame(
        {
            "Feature": X_train_scaled.columns,
            "Importance": pumpkin_classifier.feature_importances_,
        }
    )
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance_df


In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=feature_importance_df,
    x="Importance",
    y="Feature",
    hue="Feature",
    palette="viridis",
    legend=False,
)
ax.set(title="Random Forest feature importance", xlabel="Relative importance", ylabel="")
plt.tight_layout()
plt.show()


## 11. Interpretation checklist

When reporting this model, address the following questions:

1. Which class is harder to predict, based on class-specific recall and the confusion matrix?
2. Are accuracy and F1 similar? If not, what does the class distribution suggest?
3. Which features receive the greatest impurity-based importance?
4. Does strong predictive importance imply that a feature causes seed variety? Why not?
5. How might cross-validation or hyperparameter tuning provide a more stable performance estimate?

These questions move the analysis from merely fitting a model to evaluating whether its results are credible and substantively meaningful.
